Sentiment Classifier using DistilBERT + IMDB Dataset
Author: Devanshi Saxena

 Install & Import

In [ ]:
!pip install transformers datasets scikit-learn nltk -q

import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Dataset Prep

In [ ]:
# Loading IMDB reviews dataset
dataset = load_dataset("imdb")

# Using a small subset of data (training: 2k, testing: 2k)
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["test"].shuffle(seed=42).select(range(2000))


 Cleaning text: basic lowercase + stripping

In [ ]:
def clean_text(text):
    return text.lower().strip()

train_data = train_data.map(lambda x: {"text": clean_text(x["text"])})
test_data = test_data.map(lambda x: {"text": clean_text(x["text"])})

print("Sample review after cleaning:\n", train_data[0]["text"][:300], "...")


Sample review after cleaning:
 there is no relation at all between fortier and profiler but the fact that both are police series about violent crimes. profiler looks crispy, fortier looks classic. profiler plots are quite simple. fortier's plot are far more complicated... fortier looks more like prime suspect, if we have to spot  ...


 Model Load

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=512
)


Device set to use cuda:0


 Prompt Testing

In [ ]:
review_example = test_data[10]["text"]

prompts = [
    f"Classify the sentiment of this review as positive or negative: {review_example}",
    f"Is this a positive or negative review? Answer with one word. Review: {review_example}",
    f"Analyze the review and give sentiment (positive/negative): {review_example}"
]

print("\n----- Prompt Variations -----")
for i, p in enumerate(prompts, 1):
    out = classifier(p)[0]
    print(f"\nPrompt {i}: {p[:120]}...")
    print("Prediction:", out)



----- Prompt Variations -----

Prompt 1: Classify the sentiment of this review as positive or negative: if you're a fan of turkish and middle eastern music, you'...
Prediction: {'label': 'POSITIVE', 'score': 0.9997460246086121}

Prompt 2: Is this a positive or negative review? Answer with one word. Review: if you're a fan of turkish and middle eastern music...
Prediction: {'label': 'POSITIVE', 'score': 0.9997517466545105}

Prompt 3: Analyze the review and give sentiment (positive/negative): if you're a fan of turkish and middle eastern music, you're i...
Prediction: {'label': 'POSITIVE', 'score': 0.9997630715370178}


 Evaluation

In [ ]:
 y_true, y_pred = [], []

for row in test_data:
    out = classifier(row["text"], truncation=True, max_length=512)[0]
    pred_label = 1 if out["label"] == "POSITIVE" else 0
    y_pred.append(pred_label)
    y_true.append(row["label"])

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")

print("\n----- Evaluation Results -----")
print("Accuracy:", round(acc, 3))
print("Precision:", round(prec, 3))
print("Recall:", round(rec, 3))
print("F1-score:", round(f1, 3))



----- Evaluation Results -----
Accuracy: 0.891
Precision: 0.912
Recall: 0.866
F1-score: 0.888


 Troubleshooting Notes

Common Issues & Fixes:
- Overfitting on small training set → use dropout or expand training data.
- Model unstable on long reviews → truncate/pad consistently.
- Prompt wording changes predictions → try ensemble or fixed template.
